<a href="https://colab.research.google.com/github/dharani232007/CNN---Project/blob/main/RNN_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import libraries

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    SimpleRNN, LSTM, GRU, Dense,
    Embedding, Bidirectional, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Understand sequence data

sequence = np.array([10, 20, 30, 40, 50])

for time_step, value in enumerate(sequence):
    print(f"Time step {time_step}: Input = {value}")


In [ ]:
#Implement a vanilla RNN from scratch

class VanillaRNN:
    def __init__(self, input_size, hidden_size, output_size):
        rng = np.random.default_rng(42)

        self.Wxh = rng.normal(0, 0.1, (input_size, hidden_size))
        self.Whh = rng.normal(0, 0.1, (hidden_size, hidden_size))
        self.Why = rng.normal(0, 0.1, (hidden_size, output_size))

        self.bh = np.zeros(hidden_size)
        self.by = np.zeros(output_size)

    def forward(self, sequence):
        h = np.zeros(self.Whh.shape[0])
        states = []

        for x in sequence:
            h = np.tanh(x @ self.Wxh + h @ self.Whh + self.bh)
            states.append(h.copy())

        output = h @ self.Why + self.by
        return output, np.array(states)


rnn = VanillaRNN(input_size=1, hidden_size=8, output_size=1)

X = np.array([[1.0], [2.0], [3.0]])
output, hidden_states = rnn.forward(X)

print("Output:", output)
print("Hidden states shape:", hidden_states.shape)


In [ ]:
#Prepare time-series data

t = np.arange(1000)

series = (
    0.005 * t
    + np.sin(2 * np.pi * t / 30)
    + 0.2 * np.sin(2 * np.pi * t / 7)
).astype(np.float32)

plt.figure(figsize=(12, 4))
plt.plot(series)
plt.title("Synthetic Time Series")
plt.xlabel("Time")
plt.ylabel("Value")
plt.show()

In [ ]:
#Create sliding windows


def create_sequences(data, window_size=30):
    X, y = [], []

    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])

    return (
        np.array(X, dtype=np.float32)[..., np.newaxis],
        np.array(y, dtype=np.float32)
    )


X, y = create_sequences(series, window_size=30)

split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

In [ ]:
#Train a SimpleRNN

simple_rnn = Sequential([
    SimpleRNN(
        32,
        activation="tanh",
        input_shape=(30, 1)
    ),
    Dense(16, activation="relu"),
    Dense(1)
])

simple_rnn.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = simple_rnn.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
#Plot training and validation loss

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("SimpleRNN Learning Curve")
plt.legend()
plt.show()

In [ ]:
#Evaluate predictions

predictions = simple_rnn.predict(X_test, verbose=0).flatten()

mae = np.mean(np.abs(y_test - predictions))
rmse = np.sqrt(np.mean((y_test - predictions) ** 2))

print("Test MAE:", round(float(mae), 4))
print("Test RMSE:", round(float(rmse), 4))

plt.figure(figsize=(12, 4))
plt.plot(y_test[:150], label="Actual")
plt.plot(predictions[:150], label="Predicted")
plt.title("Actual vs Predicted Values")
plt.legend()
plt.show()

In [ ]:
#Compare SimpleRNN, LSTM and GRU
def build_model(model_type):
    layer_map = {
        "SimpleRNN": SimpleRNN,
        "LSTM": LSTM,
        "GRU": GRU
    }

    return Sequential([
        layer_map[model_type](32, input_shape=(30, 1)),
        Dense(16, activation="relu"),
        Dense(1)
    ])


results = {}

for model_name in ["SimpleRNN", "LSTM", "GRU"]:
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)

    model = build_model(model_name)
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])

    model.fit(
        X_train, y_train,
        epochs=30,
        batch_size=32,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0
    )

    loss, mae = model.evaluate(X_test, y_test, verbose=0)
    results[model_name] = {"MSE": loss, "MAE": mae}

for name, scores in results.items():
    print(name, scores)

In [ ]:
#Bidirectional LSTM

bidirectional_model = Sequential([
    Bidirectional(
        LSTM(32),
        input_shape=(30, 1)
    ),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1)
])

bidirectional_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

bidirectional_model.summary()

In [ ]:
#Text sentiment classification with LSTM

from tensorflow.keras.layers import TextVectorization

texts = np.array([
    "this movie is excellent",
    "i really enjoyed this film",
    "wonderful story and acting",
    "this movie is terrible",
    "i hated this film",
    "boring story and acting"
])

labels = np.array([1, 1, 1, 0, 0, 0])

vectorizer = TextVectorization(
    max_tokens=1000,
    output_mode="int",
    output_sequence_length=10
)
vectorizer.adapt(texts)

sentiment_model = Sequential([
    tf.keras.Input(shape=(), dtype=tf.string),
    vectorizer,
    Embedding(1000, 16, mask_zero=True),
    LSTM(16),
    Dense(1, activation="sigmoid")
])

sentiment_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

sentiment_model.fit(texts, labels, epochs=30, verbose=1)

sample = np.array(["excellent movie and wonderful story"])
print("Positive score:",
      sentiment_model.predict(sample, verbose=0)[0][0])


In [ ]:
#Save the trained model

simple_rnn.save("simple_rnn_forecaster.keras")

loaded_model = tf.keras.models.load_model(
    "simple_rnn_forecaster.keras"
)

print("Model saved and loaded successfully.")